# Telecom Customer Churn — Who's About to Cancel, and Why?

**Research question:** Which customers are about to cancel, and what drives them to leave?

Run each cell top to bottom. Where you see **YOUR TURN**, read the output above it and write the finding in your own words.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Load the data

**YOUR TURN:** After running the cell below, describe the dataset in one or two sentences — how many customers, how many columns, what each row represents.


In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.shape
df.head()


## 2. How bad is churn? (baseline)

Before any model, know the base rate. If 27% of customers churn, a model that just guesses 'nobody churns' is 73% accurate — and useless.

**YOUR TURN:** What's the churn rate? Write it down — every model result gets judged against this.


In [ ]:
df['Churn'].value_counts(normalize=True).round(3)


## 3. Find the dirt

**YOUR TURN:** What did you find — any missing values, any column stored as the wrong type? How did you fix it, and how many rows did the fix cost you?


In [ ]:
print(df.isnull().sum().sort_values(ascending=False).head())
print()
print(df.dtypes[df.dtypes == 'object'])


In [ ]:
# TotalCharges arrives as text (object) with a few blank strings — fix it
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().reset_index(drop=True)
df.shape


## 4. Finding 1 — Contract type

**YOUR TURN:** Which contract type churns the most? How big is the gap vs the others? In one sentence, why do you think that is?


In [ ]:
df.groupby('Contract')['Churn'].agg(['mean', 'count']).round(3)


## 5. Finding 2 — Tenure

**YOUR TURN:** How long do churned customers stick around vs customers who stay? What does that tell you about *when* the company should intervene?


In [ ]:
df[df['Churn'] == 'Yes']['tenure'].hist(alpha=0.6, label='Churned', bins=30)
df[df['Churn'] == 'No']['tenure'].hist(alpha=0.6, label='Stayed', bins=30)
plt.legend()
plt.xlabel('Tenure (months)')
plt.title('Tenure distribution by churn status')
plt.show()
print(df.groupby('Churn')['tenure'].median())


## 6. Finding 3 — Monthly charges

**YOUR TURN:** Do churners pay more or less per month? Combine this with the tenure finding — what's the story?


In [ ]:
sns.boxplot(data=df, x='Churn', y='MonthlyCharges')
plt.title('Monthly charges by churn status')
plt.show()
print(df.groupby('Churn')['MonthlyCharges'].median().round(2))


## 7. Can we predict it? Logistic Regression vs Random Forest

Baseline model vs a stronger one. Stratified 80/20 split so both sets keep the same churn rate.

**YOUR TURN:** Which model won, and on what metric? Why is accuracy the *wrong* metric to judge these models on?


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

X = df.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

lr = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
rf = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=200, random_state=42))])

for name, model in [('Logistic Regression', lr), ('Random Forest', rf)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(f'--- {name} ---')
    print(classification_report(y_test, pred, digits=3))
    print(f'ROC-AUC: {roc_auc_score(y_test, proba):.3f}')
    print()


## 8. What drives the prediction?

**YOUR TURN:** Which 3 features matter most? Do they line up with your EDA findings from sections 4–6, or did the model surprise you?


In [ ]:
importances = rf.named_steps['clf'].feature_importances_
feat_names = rf.named_steps['pre'].get_feature_names_out()
fi = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(10)
fi.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 10 features — Random Forest')
plt.tight_layout()
plt.show()


## 9. Retention playbook

**YOUR TURN:** You're presenting to the telecom's retention team. Write 3 recommendations, each one tied to a specific finding above. Example shape: 'Finding X shows ___, so we should ___.'
